In [16]:
# imports

import pickle
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import (
    when, col, unix_timestamp, round as spark_round,
    hour, dayofweek, sin, cos, lit
)

import lightgbm as lgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from scipy.stats import randint, uniform


from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve






In [2]:
# initialize spark session

try:
    spark.stop()
except:
    pass

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

spark = (
    SparkSession.builder
        .appName("Taxi Tip Regression")
        .config("spark.jars.packages", "com.microsoft.azure:synapseml_2.12:0.10.2")
        .getOrCreate()
)

print("SparkSession ready with SynapseML")


SparkSession ready with SynapseML


In [3]:
# ingest 2023 data

parquet_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
local_path = "/tmp/yellow_tripdata_2023-01.parquet"
urllib.request.urlretrieve(parquet_url, local_path)
df_spark = spark.read.parquet(local_path)

df_spark.printSchema()
df_spark.show(5)


root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

In [4]:
# add all the created features such as tip_pct, trip time, pickup hour, sin and cos hour etc

df_spark_final = (
    df_spark
    .withColumn("tip_pct",
        when(col("fare_amount") > 0, col("tip_amount") / col("fare_amount") * 100)
        .otherwise(0)
    )
    .withColumn("trip_time_minutes",
        spark_round(
            (unix_timestamp(col("tpep_dropoff_datetime")) -
             unix_timestamp(col("tpep_pickup_datetime"))) / 60,
            2
        )
    )
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))
    .withColumn("hour_sin", sin(2 * F.pi() * col("pickup_hour") / lit(24)))
    .withColumn("hour_cos", cos(2 * F.pi() * col("pickup_hour") / lit(24)))
    .withColumn("day_of_week", dayofweek(col("tpep_pickup_datetime")))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0))
    .withColumn("fare_per_min", col("fare_amount") / col("trip_time_minutes"))
    .withColumn("fare_per_mile", col("fare_amount") / col("trip_distance"))
)

df_spark_final.select(
    "tip_pct", "trip_time_minutes", "hour_sin", "hour_cos",
    "day_of_week", "is_weekend", "fare_per_min", "fare_per_mile"
).show(5)


+------------------+-----------------+--------+--------+-----------+----------+------------------+------------------+
|           tip_pct|trip_time_minutes|hour_sin|hour_cos|day_of_week|is_weekend|      fare_per_min|     fare_per_mile|
+------------------+-----------------+--------+--------+-----------+----------+------------------+------------------+
|               0.0|             8.43|     0.0|     1.0|          1|         1|1.1032028469750892| 9.587628865979383|
|50.632911392405056|             6.32|     0.0|     1.0|          1|         1|              1.25| 7.181818181818182|
|100.67114093959731|            12.75|     0.0|     1.0|          1|         1|1.1686274509803922| 5.936254980079682|
|               0.0|             9.62|     0.0|     1.0|          1|         1|1.2577962577962578| 6.368421052631579|
|28.771929824561397|            10.83|     0.0|     1.0|          1|         1|1.0526315789473684|7.9720279720279725|
+------------------+-----------------+--------+--------+

In [5]:
# apply filters

df_spark_final = df_spark_final.filter(
    (col("fare_amount") > 3) &
    (col("fare_amount") < 200) &
    (col("tip_pct") >= 0) & (col("tip_pct") <= 100) &
    (col("RatecodeID") == 1) &
    (col("trip_time_minutes") >= 0) & (col("trip_time_minutes") <= 800) &
    (col("tolls_amount") >= 0) & (col("tolls_amount") <= 50) &
    (col("trip_distance") > 0) & (col("trip_distance") < 35)
)


In [6]:
# check on data loss

prev_count = df_spark.count()
final_count = df_spark_final.count()
print(f"Rows before filter: {prev_count}, after filter: {final_count}")
print(f"Percentage retained: {final_count/prev_count*100:.2f}%")


Rows before filter: 3066766, after filter: 2785198
Percentage retained: 90.82%


In [7]:
# baseline lgb training to predict tip_pct


pdf = df_spark_final.sample(fraction=0.10, seed=42).toPandas()

num_cols = [
    "passenger_count", "trip_distance", "trip_time_minutes",
    "extra", "tolls_amount", "congestion_surcharge", "airport_fee",
    "fare_per_min", "fare_per_mile", "hour_sin", "hour_cos",
    "day_of_week", "is_weekend"
]
cat_cols = ["payment_type", "VendorID"]

X_num = pdf[num_cols]
X_cat = pd.get_dummies(pdf[cat_cols].astype(str), prefix=cat_cols)
X = pd.concat([X_num, X_cat], axis=1)
y = pdf["tip_pct"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtrain = lgb.Dataset(X_train, label=y_train)
dtest = lgb.Dataset(X_test, label=y_test, reference=dtrain)

params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.1,
    "num_leaves": 31,
    "bagging_fraction": 0.8,
    "feature_fraction": 0.8,
    "verbose": -1
}

model = lgb.train(
    params,
    dtrain,
    num_boost_round=200,
    valid_sets=[dtest],
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=10)]
)

preds = model.predict(X_test, num_iteration=model.best_iteration)
rmse = mean_squared_error(y_test, preds) ** 0.5
r2   = r2_score(y_test, preds)
print(f"Baseline RMSE = {rmse:.2f}")
print(f"Baseline R²   = {r2:.3f}")


Training until validation scores don't improve for 20 rounds
[10]	valid_0's rmse: 9.46943
[20]	valid_0's rmse: 8.76872
[30]	valid_0's rmse: 8.66502
[40]	valid_0's rmse: 8.64709
[50]	valid_0's rmse: 8.64112
[60]	valid_0's rmse: 8.63954
[70]	valid_0's rmse: 8.63967
[80]	valid_0's rmse: 8.63968
[90]	valid_0's rmse: 8.63933
[100]	valid_0's rmse: 8.63944
[110]	valid_0's rmse: 8.63909
[120]	valid_0's rmse: 8.63852
[130]	valid_0's rmse: 8.63901
[140]	valid_0's rmse: 8.63974
Early stopping, best iteration is:
[121]	valid_0's rmse: 8.63838
Baseline RMSE = 8.64
Baseline R²   = 0.605


Performance of direct regression doesn't want to pass the 8.6 RMSE barrier regardless of methods utilized, likely not a good option

In [8]:
# lgbm regression to predict just ride with tip or without tip


pdf = df_spark_final.sample(fraction=0.10, seed=42).toPandas()
pdf["did_tip"] = (pdf["tip_pct"] > 0).astype(int)

features = [
    "passenger_count", "trip_distance", "trip_time_minutes",
    "extra", "tolls_amount", "congestion_surcharge", "airport_fee",
    "fare_per_min", "fare_per_mile", "hour_sin", "hour_cos",
    "day_of_week", "is_weekend"
]
cat_feats = ["payment_type", "VendorID"]

X = pd.concat([
    pdf[features],
    pd.get_dummies(pdf[cat_feats].astype(str), prefix=cat_feats)
], axis=1)
y = pdf["did_tip"]

# 2) Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3) Initialize classifier
clf = lgb.LGBMClassifier(
    objective="binary",
    metric="binary_logloss"
)

# 4) Fit with early stopping via callback
clf.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=20)]
)

# 5) Evaluate
pred_bin = clf.predict(X_val)
print("Binary accuracy:", accuracy_score(y_val, pred_bin))
print(classification_report(y_val, pred_bin))


Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[68]	valid_0's binary_logloss: 0.116558
Binary accuracy: 0.9726750448833034
              precision    recall  f1-score   support

           0       1.00      0.87      0.93     11908
           1       0.97      1.00      0.98     43792

    accuracy                           0.97     55700
   macro avg       0.98      0.94      0.96     55700
weighted avg       0.97      0.97      0.97     55700



Despite not great performance at predicting tip amount, direct regression does seem to result in good performance when just predicting if there was a tip at all.

In [10]:
# Regression with fixed bins determined by volume

# 1) Define cutpoints (0%, 1/3, 2/3 quantiles)
q1, q2 = 24.04, 29.35

# 2) Build 4‐bucket bins:
#    0 → zero‐tip, 1 → low, 2 → medium, 3 → high
bins   = [-np.inf, 0, q1, q2, np.inf]
labels = [0, 1, 2, 3]

# 3) Apply to the full true & predicted arrays
sentiment_true = pd.cut(y_test,   bins=bins, labels=labels, include_lowest=True)
sentiment_pred = pd.cut(preds,     bins=bins, labels=labels, include_lowest=True)

# 4) Drop any NaNs, then compute accuracy
mask = sentiment_true.notna() & sentiment_pred.notna()
print("Bucket‐accuracy (4 bins incl. zero):",
      accuracy_score(sentiment_true[mask], sentiment_pred[mask]))


Bucket‐accuracy (4 bins incl. zero): 0.45535008976660685


It seems with direct regression into bins we get a poor accuracy.

In [11]:
# Attempt to find a better bucket split instead of our manually decided split



# 1) Define search grid around  q1,q2
q1_vals = np.linspace(10, 25, 15)
q2_vals = np.linspace(25, 49, 15)

best_acc, best_pair, best_dist = 0, (None, None), None

for a in q1_vals:
    for b in q2_vals:
        if b <= a:
            continue
        # 2) Define bins and labels
        bins   = [-np.inf, 0, a, b, np.inf]
        labels = [0, 1, 2, 3]

        # 3) Bin the true values and check bucket sizes
        true4 = pd.cut(y_test, bins=bins, labels=labels, include_lowest=True)
        fracs = true4.value_counts(normalize=True, sort=False)
        # Skip if any bucket has <15% of the data
        if (fracs < 0.15).any():
            continue

        # 4) Bin predictions and compute accuracy
        pred4 = pd.cut(preds, bins=bins, labels=labels, include_lowest=True)
        mask  = true4.notna() & pred4.notna()
        acc   = accuracy_score(true4[mask], pred4[mask])

        # 5) If this is the best so far, record both acc and distribution
        if acc > best_acc:
            best_acc, best_pair = acc, (a, b)
            best_dist = (fracs * 100).round(2)

# 6) Report results
print(f"Best bucket‐acc = {best_acc:.3f} at q1, q2 = {best_pair}\n")
print("Bucket distribution (% of rides in each bin):")
print(best_dist.to_frame(name='pct_of_rides'))


Best bucket‐acc = 0.555 at q1, q2 = (np.float64(19.642857142857142), np.float64(31.857142857142858))

Bucket distribution (% of rides in each bin):
         pct_of_rides
tip_pct              
0               21.38
1               16.15
2               44.66
3               17.81


Here we instead grid-search to find better cutpoints and here we see splitting up at 19.6% and 31.8% improved the accuracy marginally

In [13]:
# Direct 4-class LGBM classifier instead of regression, keeping 0% tips but dropping nonzero tips < 8% and 2% gaps around cutpoints




# 1) Sample 10%
pdf = df_spark_final.sample(fraction=0.10, seed=42).toPandas()

# 2) Drop only the nonzero tips under 8%
mask_keep = ~((pdf["tip_pct"] > 0) & (pdf["tip_pct"] < 8))
pdf = pdf[mask_keep].copy()
print(f"Rows after dropping nonzero < 8%: {len(pdf)}")

# 3) Compute the 33rd and 67th percentiles among nonzero tips (≥ 8%)
nonzero = pdf.loc[pdf["tip_pct"] > 0, "tip_pct"]
q1, q2 = np.quantile(nonzero, [1/3, 2/3])
print(f"Cutpoints for buckets (nonzero ≥ 8%): q1 = {q1:.2f}%, q2 = {q2:.2f}%")

# 3a) Drop a 2% "gray zone" around each cutpoint so low/med and med/high are well separated
gap = 2.0
gap_mask = (
    ((pdf["tip_pct"] > q1) & (pdf["tip_pct"] < q1 + gap)) |
    ((pdf["tip_pct"] > q2) & (pdf["tip_pct"] < q2 + gap))
)
pdf = pdf[~gap_mask].copy()
print(f"Rows after dropping 2% gaps around q1 & q2: {len(pdf)}")

# 4) Build the 4-class target:
#    0 = zero-tip rides,
#    1 = low (8% ≤ tip ≤ q1),
#    2 = medium (q1 < tip ≤ q2),
#    3 = high (tip > q2)
bins   = [-np.inf,  0,     q1,     q2,    np.inf]
labels = [    0  ,   1,      2,      3   ]
pdf["sent_4"] = pd.cut(
    pdf["tip_pct"],
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

# 5) Show bucket distribution (% of kept rides)
dist = pdf["sent_4"].value_counts(normalize=True).sort_index() * 100
print("\nBucket distribution (% of kept rides):")
print(dist.to_frame("pct_of_rides"))

# 6) Prepare features and one-hot encode categoricals
features = [
    "passenger_count","trip_distance","trip_time_minutes",
    "extra","tolls_amount","congestion_surcharge","airport_fee",
    "fare_per_min","fare_per_mile","hour_sin","hour_cos",
    "day_of_week","is_weekend"
]
cat_feats = ["payment_type","VendorID"]
X = pd.concat([
    pdf[features],
    pd.get_dummies(pdf[cat_feats].astype(str), prefix=cat_feats)
], axis=1)
y = pdf["sent_4"]

# 7) Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8) Train the multiclass LightGBM classifier
clf4 = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=4,
    metric="multi_logloss",
    class_weight="balanced",
    verbose=-1
)
clf4.fit(X_train, y_train)

# 9) Evaluate on the validation set
pred4 = clf4.predict(X_val)
print("\n4-class accuracy:", accuracy_score(y_val, pred4))
print(classification_report(y_val, pred4))


Rows after dropping nonzero < 8%: 271488
Cutpoints for buckets (nonzero ≥ 8%): q1 = 24.43%, q2 = 29.63%
Rows after dropping 2% gaps around q1 & q2: 223496

Bucket distribution (% of kept rides):
        pct_of_rides
sent_4              
0          26.665354
1          31.616226
2          18.953807
3          22.764613

4-class accuracy: 0.7312751677852349
              precision    recall  f1-score   support

           0       1.00      0.87      0.93     11861
           1       0.77      0.57      0.65     14176
           2       0.54      0.89      0.67      8391
           3       0.68      0.67      0.68     10272

    accuracy                           0.73     44700
   macro avg       0.75      0.75      0.73     44700
weighted avg       0.77      0.73      0.74     44700



This time we used a direct classifier to predict tips and achieved a decent accuracy after dropping tips below 8% and adding a 2% gap between classes. Still for our needs 3 distinct classes of tip amounts is likely more than we need and simplifying to just lowtip-high tip will likly be better

In [15]:
# MODEL 2 FINAL
# this time lets cut the classes down to no-tip, low tip, and high tip

# 1) Sample 10%
pdf = df_spark_final.sample(fraction=0.10, seed=42).toPandas()

# 2) Find the 50th percentile (median) among nonzero tips for the split
nonzero = pdf.loc[pdf["tip_pct"] > 0, "tip_pct"]
q = np.quantile(nonzero, 0.5)
print(f"Cutpoint for bucket (nonzero tips): median = {q:.2f}%")

# 3) Drop a 2% "gray zone" around the median split
gap = 2.0
gap_mask = ((pdf["tip_pct"] > q - gap/2) & (pdf["tip_pct"] < q + gap/2))
pdf_gap = pdf[gap_mask].copy()      # (optional: to inspect excluded data)
pdf = pdf[~gap_mask].copy()
print(f"Rows after dropping 2% gap around {q:.2f}%: {len(pdf)}")

# 4) Build the 3-class target:
#    0 = zero-tip rides,
#    1 = low tip (0 < tip ≤ q - gap/2),
#    2 = high tip (tip > q + gap/2)
bins   = [-np.inf, 0, q, np.inf]
labels = [0, 1, 2]
pdf["sent_3"] = pd.cut(
    pdf["tip_pct"],
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

# 5) Show bucket distribution (% of kept rides)
dist = pdf["sent_3"].value_counts(normalize=True).sort_index() * 100
print("\nBucket distribution (% of kept rides):")
print(dist.to_frame("pct_of_rides"))

# 6) Prepare features and one-hot encode categoricals
features = [
    "passenger_count","trip_distance","trip_time_minutes",
    "extra","tolls_amount","congestion_surcharge","airport_fee",
    "fare_per_min","fare_per_mile","hour_sin","hour_cos",
    "day_of_week","is_weekend"
]
cat_feats = ["payment_type","VendorID"]
X = pd.concat([
    pdf[features],
    pd.get_dummies(pdf[cat_feats].astype(str), prefix=cat_feats)
], axis=1)
y = pdf["sent_3"]

# 7) Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 8) Train the multiclass LightGBM classifier
clf3 = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    metric="multi_logloss",
    class_weight="balanced",
    verbose=-1
)
clf3.fit(X_train, y_train)

# 9) Evaluate on the validation set
pred3 = clf3.predict(X_val)
print("\n3-class accuracy:", accuracy_score(y_val, pred3))
print(classification_report(y_val, pred3))


Cutpoint for bucket (nonzero tips): median = 26.61%
Rows after dropping 2% gap around 26.61%: 249383

Bucket distribution (% of kept rides):
        pct_of_rides
sent_3              
0          23.897379
1          37.427571
2          38.675050

3-class accuracy: 0.793772680794755
              precision    recall  f1-score   support

           0       1.00      0.87      0.93     11923
           1       0.77      0.70      0.73     18761
           2       0.72      0.84      0.77     19193

    accuracy                           0.79     49877
   macro avg       0.83      0.80      0.81     49877
weighted avg       0.80      0.79      0.80     49877



It seems simply dropping the medium tip class increased accuracy from 70% to 79%. Given our usecase a 3-class system of sentiment proxy may be more effective. We have the option of increasing accuracy up to 89% however that would require changing the cutoff point from "low-tip" to "high-tip" to 33% which seems a bit misleading considering a 30% tip would not be colloquially considered "low" . This is as far as we got with model 2 as the SDSC cluster was not cooperative with us when we tried to expand this model to the full set

In [17]:
with open("model2_clf3.pkl", "wb") as f:
    pickle.dump(clf3, f)